In [2]:
import os

base = "/kaggle/input/datasets/pankajsomkuwar/asvspoof-2021-df"
parts = sorted(os.listdir(base))
print(parts)

['ASVspoof2021_DF_eval_part00', 'ASVspoof2021_DF_eval_part01', 'ASVspoof2021_DF_eval_part02', 'ASVspoof2021_DF_eval_part03']


In [3]:
PROTOCOL_FILE = f"{base}/ASVspoof2021_DF_eval_part00/ASVspoof2021_DF_eval/ASVspoof2021.DF.cm.eval.trl.txt"

with open(PROTOCOL_FILE) as f:
    for _ in range(5):
        print(f.readline().strip())

DF_E_2000011
DF_E_2000013
DF_E_2000024
DF_E_2000026
DF_E_2000027


In [4]:
import urllib.request
import tarfile
import os

url = "https://www.asvspoof.org/asvspoof2021/DF-keys-full.tar.gz"
urllib.request.urlretrieve(url, "DF-keys-full.tar.gz")

with tarfile.open("DF-keys-full.tar.gz") as tar:
    tar.extractall("keys")

# see what we got
for root, dirs, files in os.walk("keys"):
    for f in files:
        print(os.path.join(root, f))

/tmp/ipykernel_58/3834487274.py:9: DeprecationWarning: Python 3.14 will, by default, filter extracted tar archives and reject files or modify their metadata. Use the filter argument to control this behavior.
  tar.extractall("keys")


keys/keys/DF/README.txt
keys/keys/DF/CM/trial_metadata.txt
keys/keys/DF/CM/RawNet2/score.txt
keys/keys/DF/CM/LFCC-LCNN/score.txt
keys/keys/DF/CM/CQCC-GMM/score.txt
keys/keys/DF/CM/LFCC-GMM/score.txt


In [5]:
METADATA_FILE = "keys/keys/DF/CM/trial_metadata.txt"

with open(METADATA_FILE) as f:
    for _ in range(5):
        print(f.readline().strip())

LA_0023 DF_E_2000011 nocodec asvspoof A14 spoof notrim progress traditional_vocoder - - - -
TEF2 DF_E_2000013 low_m4a vcc2020 Task1-team20 spoof notrim eval neural_vocoder_nonautoregressive Task1 team20 FF E
TGF1 DF_E_2000024 mp3m4a vcc2020 Task2-team12 spoof notrim eval traditional_vocoder Task2 team12 FF G
LA_0043 DF_E_2000026 mp3m4a asvspoof A09 spoof notrim eval traditional_vocoder - - - -
LA_0021 DF_E_2000027 mp3m4a asvspoof A12 spoof notrim eval neural_vocoder_autoregressive - - - -


In [6]:
import pandas as pd

rows = []
with open(METADATA_FILE) as f:
    for line in f:
        parts = line.strip().split()
        filename = parts[1]
        label = parts[5]
        partition = parts[7]
        if partition == "eval":  # stick to the official eval partition
            rows.append((filename, label))

df = pd.DataFrame(rows, columns=["filename", "label"])
print(df["label"].value_counts())

N = 2000  # or 1500, 3000 — up to you
bonafide = df[df.label == "bonafide"].sample(N, random_state=42)
spoof = df[df.label == "spoof"].sample(N, random_state=42)
subset = pd.concat([bonafide, spoof]).reset_index(drop=True)
subset.head()

label
spoof       519059
bonafide     14869
Name: count, dtype: int64


,filename,label
0,DF_E_4820478,bonafide
1,DF_E_4400935,bonafide
2,DF_E_2723935,bonafide
3,DF_E_3170788,bonafide
4,DF_E_4693126,bonafide


In [7]:
import glob
import librosa
import numpy as np
from tqdm import tqdm

def find_flac(filename):
    matches = glob.glob(f"{base}/*/ASVspoof2021_DF_eval/flac/{filename}.flac")
    print("ok")
    return matches[0] if matches else None


In [21]:
def extract_features(filepath, sr=16000, n_mfcc=13):
    y, _ = librosa.load(filepath, sr=sr)
    
    frame_length = int(0.025*sr)
    hop_length = int(0.010*sr)
    
    # MFCC
    mfcc = librosa.feature.mfcc(y=y, sr=sr, n_mfcc=n_mfcc,
                                  n_fft=frame_length, hop_length=hop_length)
    
    # Time-domain
    energy = np.array([
        np.sum(np.abs(y[i:i+frame_length]**2))
        for i in range(0, len(y)-frame_length, hop_length)
    ])
    zcr = librosa.feature.zero_crossing_rate(y, frame_length=frame_length, hop_length=hop_length)
    
    # Spectral features (new)
    spectral_centroid = librosa.feature.spectral_centroid(y=y, sr=sr, n_fft=frame_length, hop_length=hop_length)
    spectral_bandwidth = librosa.feature.spectral_bandwidth(y=y, sr=sr, n_fft=frame_length, hop_length=hop_length)
    spectral_rolloff = librosa.feature.spectral_rolloff(y=y, sr=sr, n_fft=frame_length, hop_length=hop_length)
    spectral_flux = librosa.onset.onset_strength(y=y, sr=sr, hop_length=hop_length)
    
    feat_vector = np.concatenate([
        mfcc.mean(axis=1), mfcc.std(axis=1),                          # 26
        [energy.mean(), energy.std()],                                  # 2
        [zcr.mean(), zcr.std()],                                        # 2
        [spectral_centroid.mean(), spectral_centroid.std()],            # 2
        [spectral_bandwidth.mean(), spectral_bandwidth.std()],          # 2
        [spectral_rolloff.mean(), spectral_rolloff.std()],              # 2
        [spectral_flux.mean(), spectral_flux.std()],                    # 2
    ])
    return feat_vector  # total: 38 features

X, y_labels = [], []
skipped = 0

for _, row in tqdm(subset.iterrows(), total=len(subset)):
    filepath = find_flac(row["filename"])
    if filepath is None:
        skipped += 1
        continue
    try:
        feat = extract_features(filepath)
        X.append(feat)
        y_labels.append(0 if row["label"] == "bonafide" else 1)
    except Exception as e:
        skipped += 1

X = np.array(X)
y_labels = np.array(y_labels)
print(f"Extracted: {X.shape}, Skipped: {skipped}")

  0%|          | 1/4000 [00:00<08:54,  7.49it/s]/tmp/ipykernel_58/2001182635.py:2: UserWarning: PySoundFile failed. Trying audioread instead.
  y, _ = librosa.load(filepath, sr=sr)
/usr/local/lib/python3.12/dist-packages/librosa/core/audio.py:184: FutureWarning: librosa.core.audio.__audioread_load
	Deprecated as of librosa version 0.10.0.
	It will be removed in librosa version 1.0.
  y, sr_native = __audioread_load(path, offset, duration, dtype)


ok
ok


  0%|          | 2/4000 [00:00<18:14,  3.65it/s]/tmp/ipykernel_58/2001182635.py:2: UserWarning: PySoundFile failed. Trying audioread instead.
  y, _ = librosa.load(filepath, sr=sr)
/usr/local/lib/python3.12/dist-packages/librosa/core/audio.py:184: FutureWarning: librosa.core.audio.__audioread_load
	Deprecated as of librosa version 0.10.0.
	It will be removed in librosa version 1.0.
  y, sr_native = __audioread_load(path, offset, duration, dtype)


ok


  0%|          | 3/4000 [00:00<20:08,  3.31it/s]/tmp/ipykernel_58/2001182635.py:2: UserWarning: PySoundFile failed. Trying audioread instead.
  y, _ = librosa.load(filepath, sr=sr)
/usr/local/lib/python3.12/dist-packages/librosa/core/audio.py:184: FutureWarning: librosa.core.audio.__audioread_load
	Deprecated as of librosa version 0.10.0.
	It will be removed in librosa version 1.0.
  y, sr_native = __audioread_load(path, offset, duration, dtype)


...

  0%|          | 6/4000 [00:01<14:10,  4.69it/s]/tmp/ipykernel_58/2001182635.py:2: UserWarning: PySoundFile failed. Trying audioread instead.
  y, _ = librosa.load(filepath, sr=sr)
/usr/local/lib/python3.12/dist-packages/librosa/core/audio.py:184: FutureWarning: librosa.core.audio.__audioread_load
	Deprecated as of librosa version 0.10.0.
	It will be removed in librosa version 1.0.
  y, sr_native = __audioread_load(path, offset, duration, dtype)


ok


  0%|          | 9/4000 [00:01<09:27,  7.03it/s]/tmp/ipykernel_58/2001182635.py:2: UserWarning: PySoundFile failed. Trying audioread instead.
  y, _ = librosa.load(filepath, sr=sr)
/usr/local/lib/python3.12/dist-packages/librosa/core/audio.py:184: FutureWarning: librosa.core.audio.__audioread_load
	Deprecated as of librosa version 0.10.0.
	It will be removed in librosa version 1.0.
  y, sr_native = __audioread_load(path, offset, duration, dtype)
  0%|          | 9/4000 [00:01<13:08,  5.06it/s]

ok
ok
ok


KeyboardInterrupt: 

In [10]:
import pandas as pd

feature_cols = [f"feat_{i}" for i in range(X.shape[1])]
df_features = pd.DataFrame(X, columns=feature_cols)
df_features.to_csv("features.csv", index=False)

df_labels = pd.DataFrame({"label": y_labels})
df_labels.to_csv("labels.csv", index=False)

In [11]:
rows = []
with open(METADATA_FILE) as f:
    for line in f:
        parts = line.strip().split()
        filename = parts[1]
        source = parts[3]      # "asvspoof" or "vcc2020"
        attack = parts[4]      # e.g. "A14", or "-" for bonafide
        label = parts[5]
        partition = parts[7]
        if partition == "eval":
            rows.append((filename, source, attack, label))

df = pd.DataFrame(rows, columns=["filename", "source", "attack", "label"])

# Check what attack types exist and how many samples each has
print(df[(df.source == "asvspoof") & (df.label == "spoof")]["attack"].value_counts())

attack
A16    5797
A11    5788
A13    5756
A08    5740
A07    5723
A10    5709
A14    5691
A17    5654
A12    5628
A19    5627
A15    5588
A18    5578
A09    5577
Name: count, dtype: int64


In [12]:
HELD_OUT_ATTACKS = ["A16", "A09"]

# Bonafide samples (not tied to any attack type, "-" in the attack column)
bonafide_df = df[df.label == "bonafide"]

# Spoof samples split by whether their attack type is held out
spoof_train_pool = df[(df.label == "spoof") & (df.source == "asvspoof") & (~df.attack.isin(HELD_OUT_ATTACKS))]
spoof_unseen_pool = df[(df.label == "spoof") & (df.source == "asvspoof") & (df.attack.isin(HELD_OUT_ATTACKS))]

print("Bonafide available:", len(bonafide_df))
print("Spoof (seen attacks) available:", len(spoof_train_pool))
print("Spoof (unseen attacks) available:", len(spoof_unseen_pool))

Bonafide available: 14869
Spoof (seen attacks) available: 62482
Spoof (unseen attacks) available: 11374


In [13]:
N_train = 1500  # per class, for the training pool
N_unseen = 500   # per class, for the unseen-attack test pool

# Training pool: bonafide + spoof from SEEN attacks only
train_bonafide = bonafide_df.sample(N_train, random_state=42)
train_spoof = spoof_train_pool.sample(N_train, random_state=42)
train_subset = pd.concat([train_bonafide, train_spoof]).reset_index(drop=True)

# Unseen-attack test pool: bonafide + spoof from HELD-OUT attacks only
remaining_bonafide = bonafide_df.drop(train_bonafide.index)
unseen_bonafide = remaining_bonafide.sample(N_unseen, random_state=42)
unseen_spoof = spoof_unseen_pool.sample(N_unseen, random_state=42)
unseen_subset = pd.concat([unseen_bonafide, unseen_spoof]).reset_index(drop=True)

print("Train subset:", train_subset.shape)
print("Unseen-attack test subset:", unseen_subset.shape)

Train subset: (3000, 4)
Unseen-attack test subset: (1000, 4)


In [14]:
def extract_and_save(subset_df, label_suffix):
    X, y_labels = [], []
    skipped = 0
    for _, row in tqdm(subset_df.iterrows(), total=len(subset_df)):
        filepath = find_flac(row["filename"])
        if filepath is None:
            skipped += 1
            continue
        try:
            feat = extract_features(filepath)
            X.append(feat)
            y_labels.append(0 if row["label"] == "bonafide" else 1)
        except Exception:
            skipped += 1
    X = np.array(X)
    y_labels = np.array(y_labels)
    print(f"{label_suffix}: Extracted {X.shape}, Skipped {skipped}")

    feature_cols = [f"feat_{i}" for i in range(X.shape[1])]
    pd.DataFrame(X, columns=feature_cols).to_csv(f"features_{label_suffix}.csv", index=False)
    pd.DataFrame({"label": y_labels}).to_csv(f"labels_{label_suffix}.csv", index=False)

extract_and_save(train_subset, "gentrain")
extract_and_save(unseen_subset, "genunseen")

  0%|          | 0/3000 [00:00<?, ?it/s]/tmp/ipykernel_58/1686843192.py:11: UserWarning: PySoundFile failed. Trying audioread instead.
  y, _ = librosa.load(filepath, sr=sr)
/usr/local/lib/python3.12/dist-packages/librosa/core/audio.py:184: FutureWarning: librosa.core.audio.__audioread_load
	Deprecated as of librosa version 0.10.0.
	It will be removed in librosa version 1.0.
  y, sr_native = __audioread_load(path, offset, duration, dtype)
  0%|          | 2/3000 [00:00<07:07,  7.01it/s]/tmp/ipykernel_58/1686843192.py:11: UserWarning: PySoundFile failed. Trying audioread instead.
  y, _ = librosa.load(filepath, sr=sr)
/usr/local/lib/python3.12/dist-packages/librosa/core/audio.py:184: FutureWarning: librosa.core.audio.__audioread_load
	Deprecated as of librosa version 0.10.0.
	It will be removed in librosa version 1.0.
  y, sr_native = __audioread_load(path, offset, duration, dtype)
  0%|          | 3/3000 [00:00<08:14,  6.06it/s]/tmp/ipykernel_58/1686843192.py:11: UserWarning: PySoundFi

gentrain: Extracted (3000, 38), Skipped 0


  0%|          | 0/1000 [00:00<?, ?it/s]/tmp/ipykernel_58/1686843192.py:11: UserWarning: PySoundFile failed. Trying audioread instead.
  y, _ = librosa.load(filepath, sr=sr)
/usr/local/lib/python3.12/dist-packages/librosa/core/audio.py:184: FutureWarning: librosa.core.audio.__audioread_load
	Deprecated as of librosa version 0.10.0.
	It will be removed in librosa version 1.0.
  y, sr_native = __audioread_load(path, offset, duration, dtype)
  0%|          | 1/1000 [00:00<02:19,  7.17it/s]/tmp/ipykernel_58/1686843192.py:11: UserWarning: PySoundFile failed. Trying audioread instead.
  y, _ = librosa.load(filepath, sr=sr)
/usr/local/lib/python3.12/dist-packages/librosa/core/audio.py:184: FutureWarning: librosa.core.audio.__audioread_load
	Deprecated as of librosa version 0.10.0.
	It will be removed in librosa version 1.0.
  y, sr_native = __audioread_load(path, offset, duration, dtype)
  0%|          | 4/1000 [00:00<01:39, 10.00it/s]/tmp/ipykernel_58/1686843192.py:11: UserWarning: PySoundFi

genunseen: Extracted (1000, 38), Skipped 0


# LFCC Task

In [25]:
!pip install spafe -q

In [32]:
from spafe.features.lfcc import lfcc
from spafe.utils.preprocessing import SlidingWindow

def extract_features_lfcc(filepath, sr=16000, n_lfcc=13):
    y, _ = librosa.load(filepath, sr=sr)
    
    frame_length = int(0.025*sr)
    hop_length = int(0.010*sr)
    
    lfcc_feat = lfcc(
        y, fs=sr, num_ceps=n_lfcc,
        window=SlidingWindow(0.025, 0.010, "hamming")  # positional: win_len, win_hop, win_type
    )
    
    energy = np.array([
        np.sum(np.abs(y[i:i+frame_length]**2))
        for i in range(0, len(y)-frame_length, hop_length)
    ])
    zcr = librosa.feature.zero_crossing_rate(y, frame_length=frame_length, hop_length=hop_length)
    
    feat_vector = np.concatenate([
        lfcc_feat.mean(axis=0), lfcc_feat.std(axis=0),
        [energy.mean(), energy.std()],
        [zcr.mean(), zcr.std()],
    ])
    return feat_vector

In [33]:
test_file = find_flac(subset.iloc[0]["filename"])
test_feat = extract_features_lfcc(test_file)
print(test_feat.shape)

ok
(30,)


In [34]:
X_lfcc, y_labels_lfcc = [], []
skipped = 0

for _, row in tqdm(subset.iterrows(), total=len(subset)):
    filepath = find_flac(row["filename"])
    if filepath is None:
        skipped += 1
        continue
    try:
        feat = extract_features_lfcc(filepath)
        X_lfcc.append(feat)
        y_labels_lfcc.append(0 if row["label"] == "bonafide" else 1)
    except Exception as e:
        skipped += 1

X_lfcc = np.array(X_lfcc)
y_labels_lfcc = np.array(y_labels_lfcc)
print(f"Extracted: {X_lfcc.shape}, Skipped: {skipped}")

  0%|          | 0/4000 [00:00<?, ?it/s]/tmp/ipykernel_58/1851952903.py:5: UserWarning: PySoundFile failed. Trying audioread instead.
  y, _ = librosa.load(filepath, sr=sr)
/usr/local/lib/python3.12/dist-packages/librosa/core/audio.py:184: FutureWarning: librosa.core.audio.__audioread_load
	Deprecated as of librosa version 0.10.0.
	It will be removed in librosa version 1.0.
  y, sr_native = __audioread_load(path, offset, duration, dtype)


ok
ok


  0%|          | 2/4000 [00:00<07:07,  9.35it/s]/tmp/ipykernel_58/1851952903.py:5: UserWarning: PySoundFile failed. Trying audioread instead.
  y, _ = librosa.load(filepath, sr=sr)
/usr/local/lib/python3.12/dist-packages/librosa/core/audio.py:184: FutureWarning: librosa.core.audio.__audioread_load
	Deprecated as of librosa version 0.10.0.
	It will be removed in librosa version 1.0.
  y, sr_native = __audioread_load(path, offset, duration, dtype)
  0%|          | 3/4000 [00:00<08:50,  7.53it/s]/tmp/ipykernel_58/1851952903.py:5: UserWarning: PySoundFile failed. Trying audioread instead.
  y, _ = librosa.load(filepath, sr=sr)
/usr/local/lib/python3.12/dist-packages/librosa/core/audio.py:184: FutureWarning: librosa.core.audio.__audioread_load
	Deprecated as of librosa version 0.10.0.
	It will be removed in librosa version 1.0.
  y, sr_native = __audioread_load(path, offset, duration, dtype)


ok
ok


  0%|          | 4/4000 [00:00<09:56,  6.70it/s]/tmp/ipykernel_58/1851952903.py:5: UserWarning: PySoundFile failed. Trying audioread instead.
  y, _ = librosa.load(filepath, sr=sr)
/usr/local/lib/python3.12/dist-packages/librosa/core/audio.py:184: FutureWarning: librosa.core.audio.__audioread_load
	Deprecated as of librosa version 0.10.0.
	It will be removed in librosa version 1.0.
  y, sr_native = __audioread_load(path, offset, duration, dtype)
  0%|          | 6/4000 [00:00<08:07,  8.19it/s]/tmp/ipykernel_58/1851952903.py:5: UserWarning: PySoundFile failed. Trying audioread instead.
  y, _ = librosa.load(filepath, sr=sr)


...

ok
ok
ok


/tmp/ipykernel_58/1851952903.py:5: UserWarning: PySoundFile failed. Trying audioread instead.
  y, _ = librosa.load(filepath, sr=sr)
/usr/local/lib/python3.12/dist-packages/librosa/core/audio.py:184: FutureWarning: librosa.core.audio.__audioread_load
	Deprecated as of librosa version 0.10.0.
	It will be removed in librosa version 1.0.
  y, sr_native = __audioread_load(path, offset, duration, dtype)
100%|█████████▉| 3996/4000 [06:35<00:00, 11.83it/s]/tmp/ipykernel_58/1851952903.py:5: UserWarning: PySoundFile failed. Trying audioread instead.
  y, _ = librosa.load(filepath, sr=sr)
/usr/local/lib/python3.12/dist-packages/librosa/core/audio.py:184: FutureWarning: librosa.core.audio.__audioread_load
	Deprecated as of librosa version 0.10.0.
	It will be removed in librosa version 1.0.
  y, sr_native = __audioread_load(path, offset, duration, dtype)


ok
ok
ok


100%|██████████| 4000/4000 [06:36<00:00, 10.10it/s]

ok
ok
Extracted: (4000, 30), Skipped: 0


In [35]:
import pandas as pd

feature_cols = [f"feat_{i}" for i in range(X_lfcc.shape[1])]
pd.DataFrame(X_lfcc, columns=feature_cols).to_csv("features_lfcc.csv", index=False)
pd.DataFrame({"label": y_labels_lfcc}).to_csv("labels_lfcc.csv", index=False)

In [37]:
from IPython.display import FileLink
FileLink("features_lfcc.csv")

/kaggle/working/features_lfcc.csv

# CNN Task

## Extract mel-spectrograms

In [9]:
import librosa
import numpy as np

def extract_spectrogram(filepath, sr=16000, n_mels=128, max_len=128):
    y, _ = librosa.load(filepath, sr=sr)
    
    mel_spec = librosa.feature.melspectrogram(y=y, sr=sr, n_mels=n_mels,
                                                n_fft=int(0.025*sr), hop_length=int(0.010*sr))
    log_mel_spec = librosa.power_to_db(mel_spec, ref=np.max)
    
    # Pad or truncate to fixed length (time axis) so all inputs are the same shape
    if log_mel_spec.shape[1] < max_len:
        pad_width = max_len - log_mel_spec.shape[1]
        log_mel_spec = np.pad(log_mel_spec, ((0,0),(0,pad_width)), mode='constant', constant_values=-80)
    else:
        log_mel_spec = log_mel_spec[:, :max_len]
    
    return log_mel_spec

## Extract for all 4000 files

In [11]:
from tqdm import tqdm

X_spec, y_spec = [], []
skipped = 0

for _, row in tqdm(subset.iterrows(), total=len(subset)):
    filepath = find_flac(row["filename"])
    if filepath is None:
        skipped += 1
        continue
    try:
        spec = extract_spectrogram(filepath)
        X_spec.append(spec)
        y_spec.append(0 if row["label"] == "bonafide" else 1)
    except Exception:
        skipped += 1

X_spec = np.array(X_spec)
y_spec = np.array(y_spec)
print(f"Extracted: {X_spec.shape}, Skipped: {skipped}")

  0%|          | 0/4000 [00:00<?, ?it/s]/tmp/ipykernel_58/2267782388.py:5: UserWarning: PySoundFile failed. Trying audioread instead.
  y, _ = librosa.load(filepath, sr=sr)
/usr/local/lib/python3.12/dist-packages/librosa/core/audio.py:184: FutureWarning: librosa.core.audio.__audioread_load
	Deprecated as of librosa version 0.10.0.
	It will be removed in librosa version 1.0.
  y, sr_native = __audioread_load(path, offset, duration, dtype)


ok
ok


  0%|          | 2/4000 [00:00<09:11,  7.25it/s]/tmp/ipykernel_58/2267782388.py:5: UserWarning: PySoundFile failed. Trying audioread instead.
  y, _ = librosa.load(filepath, sr=sr)
/usr/local/lib/python3.12/dist-packages/librosa/core/audio.py:184: FutureWarning: librosa.core.audio.__audioread_load
	Deprecated as of librosa version 0.10.0.
	It will be removed in librosa version 1.0.
  y, sr_native = __audioread_load(path, offset, duration, dtype)
  0%|          | 3/4000 [00:00<09:31,  7.00it/s]/tmp/ipykernel_58/2267782388.py:5: UserWarning: PySoundFile failed. Trying audioread instead.
  y, _ = librosa.load(filepath, sr=sr)
/usr/local/lib/python3.12/dist-packages/librosa/core/audio.py:184: FutureWarning: librosa.core.audio.__audioread_load
	Deprecated as of librosa version 0.10.0.
	It will be removed in librosa version 1.0.
  y, sr_native = __audioread_load(path, offset, duration, dtype)


ok
ok


  0%|          | 4/4000 [00:00<09:36,  6.93it/s]/tmp/ipykernel_58/2267782388.py:5: UserWarning: PySoundFile failed. Trying audioread instead.
  y, _ = librosa.load(filepath, sr=sr)
/usr/local/lib/python3.12/dist-packages/librosa/core/audio.py:184: FutureWarning: librosa.core.audio.__audioread_load
	Deprecated as of librosa version 0.10.0.
	It will be removed in librosa version 1.0.
  y, sr_native = __audioread_load(path, offset, duration, dtype)
  0%|          | 6/4000 [00:00<08:09,  8.15it/s]

...

ok
ok
ok


/tmp/ipykernel_58/2267782388.py:5: UserWarning: PySoundFile failed. Trying audioread instead.
  y, _ = librosa.load(filepath, sr=sr)
/usr/local/lib/python3.12/dist-packages/librosa/core/audio.py:184: FutureWarning: librosa.core.audio.__audioread_load
	Deprecated as of librosa version 0.10.0.
	It will be removed in librosa version 1.0.
  y, sr_native = __audioread_load(path, offset, duration, dtype)
100%|█████████▉| 3996/4000 [05:41<00:00, 14.06it/s]/tmp/ipykernel_58/2267782388.py:5: UserWarning: PySoundFile failed. Trying audioread instead.
  y, _ = librosa.load(filepath, sr=sr)
/usr/local/lib/python3.12/dist-packages/librosa/core/audio.py:184: FutureWarning: librosa.core.audio.__audioread_load
	Deprecated as of librosa version 0.10.0.
	It will be removed in librosa version 1.0.
  y, sr_native = __audioread_load(path, offset, duration, dtype)
100%|█████████▉| 3998/4000 [05:41<00:00, 13.57it/s]

ok
ok
ok
ok


100%|██████████| 4000/4000 [05:41<00:00, 11.72it/s]

ok
Extracted: (4000, 128, 128), Skipped: 0


## Save as compressed numpy

In [12]:
np.savez_compressed("spectrograms.npz", X=X_spec, y=y_spec)
print("Saved.")

Saved.


In [13]:
from IPython.display import FileLink
FileLink("spectrograms.npz")

/kaggle/working/spectrograms.npz

## Train/val/test split + CNN model

In [14]:
from sklearn.model_selection import train_test_split

X_trainval, X_test, y_trainval, y_test = train_test_split(
    X_spec, y_spec, test_size=0.2, random_state=42, stratify=y_spec
)
X_train, X_val, y_train, y_val = train_test_split(
    X_trainval, y_trainval, test_size=0.2, random_state=42, stratify=y_trainval
)

# Normalize: scale to 0-1 range (spectrograms are in dB, roughly -80 to 0)
X_train = (X_train + 80) / 80
X_val = (X_val + 80) / 80
X_test = (X_test + 80) / 80

# Add channel dimension for CNN input: (samples, 128, 128, 1)
X_train = X_train[..., np.newaxis]
X_val = X_val[..., np.newaxis]
X_test = X_test[..., np.newaxis]

print(X_train.shape, X_val.shape, X_test.shape)

(2560, 128, 128, 1) (640, 128, 128, 1) (800, 128, 128, 1)


##  Build a simple CNN

In [15]:
import tensorflow as tf
from tensorflow.keras import layers, models

model = models.Sequential([
    layers.Conv2D(32, (3,3), activation='relu', input_shape=(128,128,1)),
    layers.MaxPooling2D((2,2)),
    layers.Conv2D(64, (3,3), activation='relu'),
    layers.MaxPooling2D((2,2)),
    layers.Conv2D(64, (3,3), activation='relu'),
    layers.MaxPooling2D((2,2)),
    layers.Flatten(),
    layers.Dense(64, activation='relu'),
    layers.Dropout(0.5),
    layers.Dense(1, activation='sigmoid')
])

model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
model.summary()

/usr/local/lib/python3.12/dist-packages/keras/src/layers/convolutional/base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)
I0000 00:00:1789679982.969297      58 gpu_device.cc:2020] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 13756 MB memory:  -> device: 0, name: Tesla T4, pci bus id: 0000:00:04.0, compute capability: 7.5
I0000 00:00:1789679982.972159      58 gpu_device.cc:2020] Created device /job:localhost/replica:0/task:0/device:GPU:1 with 13756 MB memory:  -> device: 1, name: Tesla T4, pci bus id: 0000:00:05.0, compute capability: 7.5


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d (Conv2D)                 │ (None, 126, 126, 32)   │           320 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d (MaxPooling2D)    │ (None, 63, 63, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_1 (Conv2D)               │ (None, 61, 61, 64)     │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_1 (MaxPooling2D)  │ (None, 30, 30, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_2 (Conv2D)               │ (None, 28, 28, 64)     │        36,928 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_2 (MaxPooling2D)  │ (None, 14, 14, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten (Flatten)               │ (None, 12544)          │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 64)             │       802,880 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 1)              │            65 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 858,689 (3.28 MB)

 Trainable params: 858,689 (3.28 MB)

 Non-trainable params: 0 (0.00 B)

## Train

In [16]:
history = model.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=20,
    batch_size=32
)

Epoch 1/20


2026-09-17 21:21:16.466193: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
2026-09-17 21:21:16.613950: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.


 9/80 ━━━━━━━━━━━━━━━━━━━━ 1s 15ms/step - accuracy: 0.5123 - loss: 0.6923

I0000 00:00:1789680078.735434   29227 device_compiler.h:196] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.


80/80 ━━━━━━━━━━━━━━━━━━━━ 9s 27ms/step - accuracy: 0.7344 - loss: 0.5530 - val_accuracy: 0.7469 - val_loss: 0.5548
Epoch 2/20
80/80 ━━━━━━━━━━━━━━━━━━━━ 1s 17ms/step - accuracy: 0.7691 - loss: 0.5122 - val_accuracy: 0.7625 - val_loss: 0.5182
Epoch 3/20
80/80 ━━━━━━━━━━━━━━━━━━━━ 2s 20ms/step - accuracy: 0.7699 - loss: 0.5004 - val_accuracy: 0.7625 - val_loss: 0.5077
Epoch 4/20
80/80 ━━━━━━━━━━━━━━━━━━━━ 1s 17ms/step - accuracy: 0.7719 - loss: 0.5006 - val_accuracy: 0.7656 - val_loss: 0.5224
Epoch 5/20
80/80 ━━━━━━━━━━━━━━━━━━━━ 1s 17ms/step - accuracy: 0.7758 - loss: 0.4916 - val_accuracy: 0.7656 - val_loss: 0.4940
Epoch 6/20
80/80 ━━━━━━━━━━━━━━━━━━━━ 1s 17ms/step - accuracy: 0.7879 - loss: 0.4690 - val_accuracy: 0.7703 - val_loss: 0.4841
Epoch 7/20
80/80 ━━━━━━━━━━━━━━━━━━━━ 1s 17ms/step - accuracy: 0.7891 - loss: 0.4399 - val_accuracy: 0.7766 - val_loss: 0.4560
Epoch 8/20
80/80 ━━━━━━━━━━━━━━━━━━━━ 1s 17ms/step - accuracy: 0.8055 - loss: 0.4087 - val_accuracy: 0.7672 - val_loss: 0.

In [17]:
test_loss, test_acc = model.evaluate(X_test, y_test)
print(f"CNN Test Accuracy: {test_acc:.4f}")

from sklearn.metrics import classification_report, confusion_matrix
y_pred_cnn = (model.predict(X_test) > 0.5).astype(int).flatten()
print(classification_report(y_test, y_pred_cnn, target_names=["bonafide", "spoof"]))
print(confusion_matrix(y_test, y_pred_cnn))

25/25 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - accuracy: 0.8750 - loss: 0.4587
CNN Test Accuracy: 0.8750
25/25 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step
              precision    recall  f1-score   support

    bonafide       0.91      0.84      0.87       400
       spoof       0.85      0.91      0.88       400

    accuracy                           0.88       800
   macro avg       0.88      0.88      0.87       800
weighted avg       0.88      0.88      0.87       800

[[335  65]
 [ 35 365]]


In [19]:
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix

cm = confusion_matrix(y_test, y_pred_cnn)
plt.figure(figsize=(5,4))
sns.heatmap(cm, annot=True, fmt="d", cmap="Oranges",
            xticklabels=["bonafide","spoof"], yticklabels=["bonafide","spoof"])
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.title("CNN Confusion Matrix")
plt.tight_layout()
plt.savefig("cnn_confusion.png")
plt.close()

# Also plot training history (train vs val accuracy over epochs) — visualizes the overfitting
plt.figure(figsize=(6,4))
plt.plot(history.history['accuracy'], label='Train Accuracy')
plt.plot(history.history['val_accuracy'], label='Val Accuracy')
plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.title("CNN Training History")
plt.legend()
plt.tight_layout()
plt.savefig("cnn_training_history.png")
plt.close()